# Matched-Parameter GPT-2 Baseline on OpenWebText

**Purpose:** Establish the best achievable PPL for a standard GPT-2-style
transformer at ~34M total parameters on OpenWebText, providing an
apples-to-apples comparison against the Fock-PARFLM v2.1 experiments
run from `colab_fock_multihead_openwebtext.ipynb`.

**Architecture:** Pre-LN GPT-2 decoder-only transformer.
- `d=384, n_layers=8, n_heads=6, d_ff=1536` → **~33.9M params** (tied embeddings)
- Matches the Fock-PARFLM's d=384, ~34M total param budget exactly.

**Training recipe:** Well-tuned (not constrained to the Fock recipe).
The goal is the **lowest PPL this architecture can achieve** at this param count.

**Data:** Same tokenized OpenWebText (GPT-2 BPE, 200M train + 2M val tokens)
reused from prior Fock-PARFLM experiments.

**Prediction (Kaplan scaling law):** val_loss ≈ 3.4, PPL ≈ 30–35.
See `companion_notes/Fock-PARFLM_vs_GPT-2_on_OpenWebText_Next_Steps.md` §2.

### Prerequisites
- OpenWebText tokenized and cached on Google Drive (reused from Phase 4/5)
- GPU with ≥16 GB VRAM (T4 sufficient; H100 preferred for speed)

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────
# Primary config: matched to Fock-PARFLM total param count (~34M).
D_MODEL     = 384
N_LAYERS    = 8
N_HEADS     = 6           # d_head = 64
D_FF        = 4 * D_MODEL # 1536 (standard 4x expansion)
MAX_LEN     = 1024        # GPT-2 natural context length
DROPOUT     = 0.0         # no dropout for small models (hurts more than helps)
TIE_EMBEDDINGS = True     # tied input/output embeddings (standard GPT-2)
VOCAB_SIZE  = 50257       # GPT-2 BPE vocabulary

# Training — well-tuned for this model size (not constrained to Fock recipe)
LR            = 6e-4      # nanoGPT default for GPT-2 Small; appropriate for ~34M
LR_MIN        = 6e-5      # 10% of peak (nanoGPT default)
WEIGHT_DECAY  = 0.1       # standard for transformers
WARMUP_STEPS  = 2000
TOTAL_STEPS   = 100_000   # ~6.5B tokens at eff_batch ~65K tokens/step
GRAD_CLIP     = 1.0
BETA1         = 0.9
BETA2         = 0.95
EVAL_INTERVAL = 2000      # matches Fock notebook
EVAL_ITERS    = 40        # matches Fock notebook
LOG_INTERVAL  = 200
CKPT_INTERVAL = 25_000
SEED          = 0

# Data — same tokenized OWT as Fock experiments
MAX_TRAIN_TOKENS = 200_000_000
VAL_TOKENS       = 2_000_000

# Block size for evaluation comparison:
# Primary eval uses MAX_LEN (1024) to give the transformer its best shot.
# A secondary eval at BLOCK_SIZE_MATCHED=512 provides the direct Fock comparison.
BLOCK_SIZE_MATCHED = 512

print(f'Config: d={D_MODEL} L={N_LAYERS} heads={N_HEADS} d_ff={D_FF}')
print(f'  max_len={MAX_LEN} dropout={DROPOUT} tie_emb={TIE_EMBEDDINGS}')
print(f'  LR={LR} warmup={WARMUP_STEPS} steps={TOTAL_STEPS}')
print(f'  weight_decay={WEIGHT_DECAY} grad_clip={GRAD_CLIP}')

In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    GDRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_matched_gpt2_baseline_owt')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    DATA_DIR    = GDRIVE_ROOT / 'data'
    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    DATA_DIR.mkdir(exist_ok=True)
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'transformers', 'huggingface_hub', 'pyarrow'])
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    CKPT_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'gpt2_baseline' / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'gpt2_baseline'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name()}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Data loading (reuse cached OWT from Fock experiments) ─
CHUNK_SIZE = 50_000

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

for alt_name in [
    'semsimula_fock_multihead_openwebtext_xi5_mh4_ob',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total: {total:,} tokens from {n_docs:,} docs ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Saved: train={len(train_ids):,}  val={len(val_ids):,}')
    del all_ids
    gc.collect()

print(f'Train tokens: {len(train_ids):,}')
print(f'Val tokens:   {len(val_ids):,}')

In [ ]:
# ── Cell 3: GPT-2 Model Definition ────────────────────────────────


class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_heads, max_len, dropout=0.0):
        super().__init__()
        assert d % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d // n_heads
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer('mask',
            torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        att = (q @ k.transpose(-2, -1)) * (self.d_head ** -0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        out = (att @ v).transpose(1, 2).reshape(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, d, d_ff, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, d, n_heads, d_ff, max_len, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = CausalSelfAttention(d, n_heads, max_len, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = MLP(d, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT2(nn.Module):
    def __init__(self, vocab_size, d, n_layers, n_heads, d_ff, max_len,
                 dropout=0.0, tie_embeddings=True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Embedding(max_len, d)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d, n_heads, d_ff, max_len, dropout) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d)
        self.tie_embeddings = tie_embeddings
        if not tie_embeddings:
            self.lm_head = nn.Linear(d, vocab_size, bias=False)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
            elif isinstance(m, nn.LayerNorm):
                torch.nn.init.ones_(m.weight)
                torch.nn.init.zeros_(m.bias)
        # GPT-2 scaling: scale residual projections by 1/sqrt(2*n_layers)
        for block in self.blocks:
            torch.nn.init.normal_(block.attn.proj.weight, mean=0.0,
                                  std=0.02 / math.sqrt(2 * len(self.blocks)))
            torch.nn.init.normal_(block.mlp.fc2.weight, mean=0.0,
                                  std=0.02 / math.sqrt(2 * len(self.blocks)))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        if self.tie_embeddings:
            logits = x @ self.tok_emb.weight.T
        else:
            logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.tok_emb.weight.numel()
            n -= self.pos_emb.weight.numel()
        return n


print('GPT2 model class defined.')

In [ ]:
# ── Cell 4: Build model + auto batch size ─────────────────────────
model = GPT2(
    vocab_size=VOCAB_SIZE, d=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    d_ff=D_FF, max_len=MAX_LEN, dropout=DROPOUT, tie_embeddings=TIE_EMBEDDINGS,
).to(DEVICE)

n_total = model.num_params()
n_non_emb = model.num_params(non_embedding=True)
print(f'GPT-2 Baseline: {n_total:,} total params  ({n_non_emb:,} non-embedding)')
print(f'  d={D_MODEL}  L={N_LAYERS}  heads={N_HEADS}  d_ff={D_FF}')
print(f'  tok_emb: {model.tok_emb.weight.numel():,}')
print(f'  pos_emb: {model.pos_emb.weight.numel():,}')
print(f'  blocks:  {sum(p.numel() for b in model.blocks for p in b.parameters()):,}')

# Auto batch sizing — find largest batch that fits
rng = np.random.default_rng(42)
BATCH_SIZE = 4
if DEVICE in ('cuda', 'mps'):
    for bs in [64, 48, 32, 24, 16, 12, 8, 4]:
        try:
            n = len(train_ids) - MAX_LEN - 1
            starts = rng.integers(0, n, size=bs)
            _x = torch.tensor(np.stack([train_ids[s:s+MAX_LEN] for s in starts]).astype(np.int64)).to(DEVICE)
            _y = torch.tensor(np.stack([train_ids[s+1:s+1+MAX_LEN] for s in starts]).astype(np.int64)).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            model.zero_grad(set_to_none=True)
            del _x, _y, _loss
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            BATCH_SIZE = bs
            break
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                del _x, _y
                if DEVICE == 'cuda':
                    torch.cuda.empty_cache()
                continue
            raise

EFFECTIVE_BATCH = BATCH_SIZE * MAX_LEN
print(f'\nBatch size: {BATCH_SIZE}  (seq_len={MAX_LEN})')
print(f'Tokens per step: {EFFECTIVE_BATCH:,}')
print(f'Total steps: {TOTAL_STEPS:,}')
print(f'Total tokens seen: {TOTAL_STEPS * EFFECTIVE_BATCH / 1e9:.2f}B')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────

def get_batch(ids, batch_size, block_size, rng):
    n = len(ids) - block_size - 1
    starts = rng.integers(0, n, size=batch_size)
    x = np.stack([ids[s:s + block_size] for s in starts]).astype(np.int64)
    y = np.stack([ids[s+1:s+1 + block_size] for s in starts]).astype(np.int64)
    return x, y


def lr_schedule(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR_MIN + 0.5 * (LR - LR_MIN) * (1.0 + math.cos(math.pi * min(progress, 1.0)))


@torch.no_grad()
def evaluate(block_size):
    model.eval()
    losses = []
    _rng = np.random.default_rng(42)
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, block_size, _rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        _, loss = model(x[:, :block_size], y[:, :block_size])
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


# Optimizer — only apply weight decay to 2D params (weights, not biases/LN)
decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
nodecay_params = [p for n, p in model.named_parameters() if p.dim() < 2 and p.requires_grad]
optim_groups = [
    {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
    {'params': nodecay_params, 'weight_decay': 0.0},
]
optimizer = torch.optim.AdamW(optim_groups, lr=LR, betas=(BETA1, BETA2), fused=(DEVICE == 'cuda'))
print(f'Optimizer: AdamW  decay_params={sum(p.numel() for p in decay_params):,}'
      f'  nodecay_params={sum(p.numel() for p in nodecay_params):,}')

# Resume from checkpoint if available
start_step = 0
best_val_ppl = float('inf')
_ckpt_path = CKPT_DIR / 'gpt2_baseline_best.pt'
if _ckpt_path.exists():
    _ckpt = torch.load(_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(_ckpt['model_state_dict'])
    if 'optimizer_state_dict' in _ckpt:
        optimizer.load_state_dict(_ckpt['optimizer_state_dict'])
        print('  Optimizer state restored.')
    start_step = _ckpt.get('step', 0)
    best_val_ppl = _ckpt.get('val_ppl', float('inf'))
    print(f'Resuming from step {start_step:,}, best PPL: {best_val_ppl:.2f}')
    del _ckpt

# Training
torch.manual_seed(SEED)
np.random.seed(SEED)
train_rng = np.random.default_rng(SEED)

log_path = RESULTS_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

model.train()
t0 = time.time()
run_loss = 0.0
n_run = 0

print(f'\n{"="*60}')
print(f'Training: steps {start_step + 1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE}  block={MAX_LEN}  lr={LR}  warmup={WARMUP_STEPS}')
print(f'  d={D_MODEL}  L={N_LAYERS}  params={n_total:,}')
print(f'{"="*60}\n')

for step in range(start_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now

    xb, yb = get_batch(train_ids, BATCH_SIZE, MAX_LEN, train_rng)
    x = torch.from_numpy(xb).to(DEVICE)
    y = torch.from_numpy(yb).to(DEVICE)

    _, loss = model(x, y)
    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    run_loss += loss.item()
    n_run += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_loss = run_loss / n_run
        elapsed = time.time() - t0
        remaining = elapsed / (step + 1 - start_step) * (TOTAL_STEPS - step - 1)
        print(f'step {step+1:>7d}/{TOTAL_STEPS}  loss={avg_loss:.4f}  '
              f'lr={lr_now:.2e}  grad={float(grad_norm):.2f}  '
              f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)')
        log_f.write(json.dumps({
            'step': step + 1, 'train_loss': avg_loss, 'lr': lr_now,
            'grad_norm': float(grad_norm), 'elapsed_sec': elapsed,
        }) + '\n')
        log_f.flush()
        run_loss = 0.0
        n_run = 0

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss_1024 = evaluate(MAX_LEN)
        val_ppl_1024 = math.exp(val_loss_1024)
        val_loss_512 = evaluate(BLOCK_SIZE_MATCHED)
        val_ppl_512 = math.exp(val_loss_512)
        is_best = val_ppl_1024 < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl_1024
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  '
              f'loss@1024={val_loss_1024:.4f} ppl@1024={val_ppl_1024:.2f}  '
              f'loss@512={val_loss_512:.4f} ppl@512={val_ppl_512:.2f}  '
              f'best={best_val_ppl:.2f}  {marker}  ({elapsed:.0f}s)')
        log_f.write(json.dumps({
            'step': step + 1, 'val_loss_1024': val_loss_1024,
            'val_ppl_1024': val_ppl_1024, 'val_loss_512': val_loss_512,
            'val_ppl_512': val_ppl_512, 'best_ppl': best_val_ppl,
        }) + '\n')
        log_f.flush()
        if is_best:
            ckpt = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'step': step + 1, 'val_loss': val_loss_1024,
                'val_ppl': val_ppl_1024, 'val_ppl_512': val_ppl_512,
                'config': {
                    'd': D_MODEL, 'n_layers': N_LAYERS, 'n_heads': N_HEADS,
                    'd_ff': D_FF, 'max_len': MAX_LEN, 'vocab_size': VOCAB_SIZE,
                    'dropout': DROPOUT, 'tie_embeddings': TIE_EMBEDDINGS,
                    'lr': LR, 'total_steps': TOTAL_STEPS, 'batch_size': BATCH_SIZE,
                },
            }
            torch.save(ckpt, CKPT_DIR / 'gpt2_baseline_best.pt')
            print(f'  Checkpoint saved (PPL={val_ppl_1024:.2f})')

    if (step + 1) % CKPT_INTERVAL == 0:
        ckpt = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'step': step + 1, 'val_ppl': val_ppl_1024 if (step + 1) % EVAL_INTERVAL == 0 else None,
            'config': {
                'd': D_MODEL, 'n_layers': N_LAYERS, 'n_heads': N_HEADS,
                'd_ff': D_FF, 'max_len': MAX_LEN, 'vocab_size': VOCAB_SIZE,
            },
        }
        torch.save(ckpt, CKPT_DIR / f'gpt2_baseline_step{step+1}.pt')
        print(f'  Periodic checkpoint saved: step {step+1}')

log_f.close()
print(f'\nTraining complete. Best PPL@1024: {best_val_ppl:.2f}')

In [ ]:
# ── Cell 6: Final comparison summary ──────────────────────────────
# Load training log and produce summary

log_data = []
with open(RESULTS_DIR / 'training_log.jsonl') as f:
    for line in f:
        log_data.append(json.loads(line))

eval_points = [r for r in log_data if 'val_ppl_1024' in r]
best_eval = min(eval_points, key=lambda r: r['val_ppl_1024'])

print('=' * 60)
print('MATCHED GPT-2 BASELINE — FINAL RESULTS')
print('=' * 60)
print(f'\nArchitecture: GPT-2 (pre-LN)')
print(f'  d={D_MODEL}  n_layers={N_LAYERS}  n_heads={N_HEADS}  d_ff={D_FF}')
print(f'  Total params:       {n_total:,}')
print(f'  Non-embedding:      {n_non_emb:,}')
print(f'  Tied embeddings:    {TIE_EMBEDDINGS}')
print(f'\nTraining:')
print(f'  LR={LR}  weight_decay={WEIGHT_DECAY}  batch={BATCH_SIZE}')
print(f'  Total steps:        {TOTAL_STEPS:,}')
print(f'  Total tokens:       {TOTAL_STEPS * EFFECTIVE_BATCH / 1e9:.2f}B')
print(f'\nBest result (step {best_eval["step"]:,}):')
print(f'  val_loss @1024:     {best_eval["val_loss_1024"]:.4f}')
print(f'  val_PPL  @1024:     {best_eval["val_ppl_1024"]:.2f}')
print(f'  val_loss @512:      {best_eval["val_loss_512"]:.4f}')
print(f'  val_PPL  @512:      {best_eval["val_ppl_512"]:.2f}')
print(f'\nKaplan prediction:    PPL ≈ 30–35  (val_loss ≈ 3.4–3.5)')

print(f'\n{"─"*60}')
print(f'Comparison with Fock-PARFLM v2.1 (~34M, same OWT data):')
print(f'  Fock best PPL @512:  207.68  (step 104K)')
print(f'  GPT-2 PPL @512:     {best_eval["val_ppl_512"]:.2f}  (step {best_eval["step"]:,})')
print(f'  GPT-2 PPL @1024:    {best_eval["val_ppl_1024"]:.2f}  (step {best_eval["step"]:,})')
ratio = 207.68 / best_eval['val_ppl_512']
print(f'  Ratio (Fock / GPT-2 @512): {ratio:.1f}x')
print(f'{"─"*60}')

summary = {
    'model': 'GPT-2 baseline',
    'params_total': n_total, 'params_non_emb': n_non_emb,
    'd': D_MODEL, 'n_layers': N_LAYERS, 'n_heads': N_HEADS,
    'lr': LR, 'total_steps': TOTAL_STEPS, 'batch_size': BATCH_SIZE,
    'best_step': best_eval['step'],
    'best_val_loss_1024': best_eval['val_loss_1024'],
    'best_val_ppl_1024': best_eval['val_ppl_1024'],
    'best_val_loss_512': best_eval['val_loss_512'],
    'best_val_ppl_512': best_eval['val_ppl_512'],
    'fock_best_ppl_512': 207.68,
    'ratio_fock_to_gpt2': ratio,
}
with open(RESULTS_DIR / 'gpt2_baseline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved to {RESULTS_DIR / "gpt2_baseline_summary.json"}')